# Arein PA Pipeline — Colab Run

Runs the prior-authorization extraction pipeline against real Llama-3.3-70B via Groq.

**Before you start:**
1. Have your Groq API key handy (starts with `gsk_...`). Get one at https://console.groq.com → API Keys.
2. Make sure `Sample_PsO_ADS_Track/` and `PA_Business_Rules.xlsx` are in your Drive — you'll be prompted for the paths in Cell 3.

**Runtime:** Default CPU is fine. GPU not needed (all LLM calls are remote).

## Cell 1 — Clone the repo from GitHub

In [ ]:
!git clone https://github.com/aryan-c0des/ADS_TT_Hackathon.git /content/ADS_TT_Hackathon
%cd /content/ADS_TT_Hackathon
!git log --oneline -5

## Cell 2 — Mount Google Drive

Click the auth link Colab shows you, sign in, paste the code back.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 3 — Locate source data in Drive and symlink into /content/

**EDIT the two paths below** to match where you've stored the data in Drive. The defaults assume a folder named `Arein - hackathon` at the root of MyDrive.

If you don't know the path, run `!ls '/content/drive/MyDrive/'` in a scratch cell first to browse.

In [ ]:
import os
from pathlib import Path

# ⬇⬇⬇  UPDATE THESE TWO LINES if your Drive layout differs  ⬇⬇⬇
DRIVE_PDFS = Path("/content/drive/MyDrive/Arein - hackathon/Sample_PsO_ADS_Track")
DRIVE_XLSX = Path("/content/drive/MyDrive/Arein - hackathon/PA_Business_Rules.xlsx")
# ⬆⬆⬆  UPDATE THESE TWO LINES if your Drive layout differs  ⬆⬆⬆

assert DRIVE_PDFS.exists(), f"PDFs not found at {DRIVE_PDFS} — fix the path above"
assert DRIVE_XLSX.exists(), f"XLSX not found at {DRIVE_XLSX} — fix the path above"

# config.py expects PDFs at REPO_ROOT/Sample_PsO_ADS_Track and the xlsx at
# REPO_ROOT/PA_Business_Rules.xlsx where REPO_ROOT = parent-of-project.
# Project is at /content/ADS_TT_Hackathon, so REPO_ROOT = /content.
for link, target in [
    ("/content/Sample_PsO_ADS_Track", DRIVE_PDFS),
    ("/content/PA_Business_Rules.xlsx", DRIVE_XLSX),
]:
    if os.path.lexists(link):
        os.remove(link)
    os.symlink(target, link)

n_pdfs = len(list(Path("/content/Sample_PsO_ADS_Track").glob("*.pdf")))
print(f"Found {n_pdfs} PDFs and the rules xlsx — expected 70 PDFs.")
assert n_pdfs == 70, f"Expected 70 PDFs, got {n_pdfs}"

## Cell 4 — Install dependencies

`pdftotext` (poppler) is needed for PDF text extraction. The Python deps are small: Groq client, pandas, openpyxl, Jinja2, matplotlib.

In [ ]:
!apt-get install -y -qq poppler-utils
!pip install -q groq pandas openpyxl jinja2 matplotlib
!pdftotext -v 2>&1 | head -1
import groq; print(f"groq sdk: {groq.__version__}")

## Cell 5 — Set the Groq API key

Paste your `gsk_...` key into the prompt. It's not echoed to the cell output and not saved to disk.

In [ ]:
from getpass import getpass
import os

os.environ["GROQ_API_KEY"] = getpass("Paste your Groq API key (gsk_...): ").strip()
key = os.environ["GROQ_API_KEY"]
assert key.startswith("gsk_"), "Key should start with gsk_ — paste again"
print(f"Key set ({len(key)} chars, prefix={key[:8]}...)")

## Cell 6 — 3-row sanity check

Wipes the synthetic seed cache so every call hits real Llama, then processes the first 3 rows. Should take ~30 seconds and make 9 Groq calls.

**What to look for in the output:**
- `[ok]` lines for each row (not `[fail]`)
- Layout detected sensibly (`single_drug` / `multi_drug` / `mega_formulary`)
- Step counts and Access Scores populated
- NO `WARNING: N cache reads came from synthetic seeds` at the end — that means real Llama responded

In [ ]:
# Wipe synthetic seeds — real LLM calls only from here
!rm -f data/llm_cache/*.json

import sys
sys.path.insert(0, '/content/ADS_TT_Hackathon')

from src import pipeline
df = pipeline.run_all(limit=3, verbose=True)
df

## Cell 7 — Render the 3 audit cards and preview one

Each card shows the extracted values next to the evidence snippets, the step graph trace, validation flags, and the score waterfall.

In [ ]:
from src import evidence_report
paths = evidence_report.render_all()
evidence_report.render_index()
print(f"Rendered {len(paths)} audit cards. Files:")
for p in paths:
    print(f"  {p}")

# Inline preview of the first card
from IPython.display import HTML
HTML(filename=str(paths[0]))

## Cell 8 — Full 79-row run (only if Cell 6 looked good)

DON'T run this until you've eyeballed the 3-row output and the audit card. This takes 5–10 minutes (~228 more Groq calls; the 9 already-cached rows are free).

The cache is preserved — only cache misses make new calls. So if you fix a prompt and re-run, only affected rows hit the LLM.

In [ ]:
df_full = pipeline.run_all(verbose=True)
print(f"\n\nFinal CSV: {len(df_full)} rows")
df_full

## Cell 9 — Render all audit cards + heatmap + score distribution

In [ ]:
from src import evidence_report, visualize
evidence_report.render_all()
evidence_report.render_index()
visualize.render_heatmap()
visualize.render_score_distribution()

from IPython.display import Image, display
display(Image("output/heatmap.png"))
display(Image("output/score_distribution.png"))

## Cell 10 — Save outputs back to Drive so they survive Colab disconnect

In [ ]:
import shutil
from pathlib import Path

DRIVE_OUT = Path("/content/drive/MyDrive/Arein - hackathon/output_colab")
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

# Copy output/ tree to Drive
for src in Path("output").rglob("*"):
    if src.is_file():
        rel = src.relative_to("output")
        dst = DRIVE_OUT / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)

# Also stash the LLM cache so you can re-run for free across Colab sessions
CACHE_BACKUP = Path("/content/drive/MyDrive/Arein - hackathon/llm_cache_colab")
CACHE_BACKUP.mkdir(parents=True, exist_ok=True)
for src in Path("data/llm_cache").glob("*.json"):
    shutil.copy2(src, CACHE_BACKUP / src.name)

print(f"Outputs saved to {DRIVE_OUT}")
print(f"LLM cache saved to {CACHE_BACKUP}")